# 🚀 Takatsuki-150M SLM Pre-Training (Kaggle T4 GPU)

Pre-trains **Takatsuki-150M** from scratch on 2x NVIDIA T4 GPUs with FP16 Mixed Precision.

In [ ]:
# 1. Check GPU
!nvidia-smi
!pip install -q torch transformers tokenizers datasets accelerate einops pyyaml tqdm matplotlib

In [ ]:
# 2. Clone repository
!rm -rf slm-lab
!git clone https://github.com/neverlone/slm-lab.git
%cd slm-lab

In [ ]:
# 3. Stream data and train 32k tokenizer
!python scripts/prepare_takatsuki_data.py --samples_per_source 25000

In [ ]:
# 4. High-Speed FP16 GPU Training Run
import os, sys, math, time
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
import matplotlib.pyplot as plt
from src.model.transformer import SLMForCausalLM, ModelArgs
from src.dataset.dataset import create_dataloader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

MODEL_CONFIG_150M = {
    'dim': 768,
    'n_layers': 12,
    'n_heads': 12,
    'n_kv_heads': 4,
    'vocab_size': 32768,
    'max_seq_len': 2048,
    'rope_theta': 10000.0,
    'norm_eps': 1e-5,
    'tie_word_embeddings': True
}

args = ModelArgs(**MODEL_CONFIG_150M)
model = SLMForCausalLM(args).to(device)
print(f'Takatsuki-150M initialized with {model.count_parameters():,} parameters.')

BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
MAX_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 500
MAX_STEPS = 20000
SAVE_INTERVAL = 1000

optimizer = AdamW(model.parameters(), lr=MAX_LR, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()
dataloader = create_dataloader(bin_path='data/tokenized/train.bin', batch_size=BATCH_SIZE, seq_len=args.max_seq_len)
data_iter = iter(dataloader)

def get_lr(it):
    if it < WARMUP_STEPS:
        return MAX_LR * (it + 1) / (WARMUP_STEPS + 1)
    if it > MAX_STEPS:
        return MIN_LR
    decay_ratio = (it - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

os.makedirs('checkpoints/kaggle', exist_ok=True)
loss_history = []
t0 = time.time()
model.train()

print('🚀 Starting Training Loop...')
for step in range(MAX_STEPS):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM_STEPS):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with autocast(dtype=torch.float16):
            _, loss, _ = model(x, labels=y)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_loss += loss.item() * GRAD_ACCUM_STEPS

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    loss_history.append(accum_loss)

    if step % 20 == 0:
        dt = time.time() - t0
        t0 = time.time()
        tok_processed = BATCH_SIZE * GRAD_ACCUM_STEPS * args.max_seq_len * 20
        tok_per_sec = tok_processed / max(dt, 1e-4)
        pct = (step / MAX_STEPS) * 100
        print(f'[Step {step:05d}/{MAX_STEPS:05d} ({pct:4.1f}%)] Loss: {accum_loss:.4f} | LR: {lr:.2e} | Speed: {tok_per_sec:,.0f} tok/s')

    if step > 0 and (step % SAVE_INTERVAL == 0 or step == MAX_STEPS - 1):
        ckpt_file = f'checkpoints/kaggle/takatsuki_step_{step}.pt'
        torch.save({'step': step, 'model_state_dict': model.state_dict(), 'loss': accum_loss, 'args': args}, ckpt_file)
        print(f'--> Saved checkpoint: {ckpt_file}')

print('✅ Pre-training completed!')

In [ ]:
# 5. Test Inference
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_file='data/tokenizer/takatsuki_tokenizer.json')
model.eval()
prompt = '<|im_start|>user\nWhat is the essence of courage in one sentence?<|im_end|>\n<|im_start|>assistant\n'
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=64, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output_ids[0].tolist()))